In [ ]:
import csv
import logging
import os
import json
import random
import re
import shutil
import subprocess
import time
import threading
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Set, Dict

import undetected_chromedriver as uc
from dotenv import load_dotenv
from selenium.webdriver.common.by import By
from openai import OpenAI

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)

load_dotenv()

BASE_DIR = Path("/home/kongla/Documents/GitHub/Real-estate-Scraping")
OUTPUT_PATH = BASE_DIR / "facebook" / "facebook_output.csv"
PROFILE_PATH = BASE_DIR / "chrome_profile"

TARGET_POSTS = 100
MAX_STAGNANT = 10
SCROLL_SIZE = 2500

MODEL_NAME = "typhoon-v2.5-30b-a3b-instruct"
MAX_WORKERS = 5
LLM_TIMEOUT = 60.0

client = OpenAI(api_key=os.getenv("TYPHOON_API_KEY"), base_url="https://api.opentyphoon.ai/v1", timeout=LLM_TIMEOUT)

GROUP_URLS: List[str] = [
    "https://www.facebook.com/share/g/17yvNkWcJG/",
    "https://www.facebook.com/share/g/17cuAKX5Py/",
    "https://www.facebook.com/share/g/1Wm4GxnHFm/",
    "https://www.facebook.com/share/g/1Dp8rpYxmE/",
    "https://www.facebook.com/share/g/1F3cnPDmGU/",
    "https://www.facebook.com/share/g/1N2eXd6ge5/",
    "https://www.facebook.com/share/g/1AnUsnRNod/",
    "https://www.facebook.com/share/g/1JyWvsFtu6/",
    "https://www.facebook.com/share/g/1AzBzdPhEL/",
    "https://www.facebook.com/share/g/18NsbCsA52/",
    "https://www.facebook.com/share/g/1CKirSUbzs/",
    "https://www.facebook.com/share/g/173xAigqGb/",
    "https://www.facebook.com/share/g/17yZmR3H1U/",
    "https://www.facebook.com/share/g/1EV9v3JxPX/",
    "https://www.facebook.com/share/g/1BXjpfQN97/",
    "https://www.facebook.com/share/g/18Mh4scEdr/",
    "https://www.facebook.com/share/g/17rf5Ppt8a/",
    "https://www.facebook.com/share/g/1EGJYxCSeT/",
    "https://www.facebook.com/share/g/1C9S5cRLgw/",
    "https://www.facebook.com/share/g/1FgHtiV8oF/",
    "https://www.facebook.com/groups/485019775925280/",
    "https://www.facebook.com/share/g/1AgeQ1WpzA/",
    "https://www.facebook.com/share/g/1XYiLFYJgX/",
    "https://www.facebook.com/share/g/1CAAbCJJVT/",
    "https://www.facebook.com/share/g/1879ZUrbpH/",
    "https://www.facebook.com/share/g/1Bhed5kut5/",
    "https://www.facebook.com/share/g/1J75CA6Aox/",
    "https://www.facebook.com/share/g/1CVwGydoVb/",
    "https://www.facebook.com/share/g/1SKVJR95jx/",
    "https://www.facebook.com/share/g/1Di1RbGM5Z/",
    "https://www.facebook.com/share/g/1AgFYb1qH7/",
    "https://www.facebook.com/share/g/1WhmM13MbK/",
    "https://www.facebook.com/share/g/1Ch6CSbFLq/",
    "https://www.facebook.com/share/g/19vVjxviuP/",
    "https://www.facebook.com/share/g/189YXg4LZr/",
    "https://www.facebook.com/share/g/1HmmyqcGTq/",
    "https://www.facebook.com/share/g/18E2JPkYMH/",
    "https://www.facebook.com/share/g/1ArRmRHKEy/",
    "https://www.facebook.com/share/g/19c9HUsnBN/",
    "https://www.facebook.com/share/g/1GTM17dYew/",
    "https://www.facebook.com/share/g/189gkv2NVu/",
    "https://www.facebook.com/share/g/1BGgqYXFrJ/",
    "https://www.facebook.com/share/g/1E2rpaskGy/",
    "https://www.facebook.com/share/g/17iWwCY6Pc/",
    "https://www.facebook.com/share/g/1biieSTrft/",
    "https://www.facebook.com/share/g/1DmTmzs5Dn/",
    "https://www.facebook.com/share/g/1GDtgBoYjd/",
    "https://www.facebook.com/share/g/17o4QPYZfr/",
    "https://www.facebook.com/share/g/1AwtrUdMkw/",
    "https://www.facebook.com/share/g/1DFcTDtguE/",
    "https://www.facebook.com/share/g/14cTZ1dxvTo/",
    "https://www.facebook.com/share/g/1ENBMKVPrw/",
    "https://www.facebook.com/share/g/1Dksryieym/",
    "https://www.facebook.com/share/g/1DAyY5idaF/",
    "https://www.facebook.com/share/g/1CNm4wbqSe/",
    "https://www.facebook.com/share/g/17uw8vv7vb/",
    "https://www.facebook.com/share/g/1GPMEpyZEa/",
    "https://www.facebook.com/share/g/1DEHfe3cS3/",
    "https://www.facebook.com/share/g/1GZhnbeEVJ/",
    "https://www.facebook.com/share/g/1CUhdjzY2y/",
    "https://www.facebook.com/share/g/17G11FQ4MS/",
    "https://www.facebook.com/share/g/1Kmt1wQmrZ/",
    "https://www.facebook.com/share/g/1MhTuADusV/",
    "https://www.facebook.com/share/g/1CTofFNtcq/",
    "https://www.facebook.com/share/g/18A2nRTGhd/",
    "https://www.facebook.com/share/g/1AsSXshu4s/",
    "https://www.facebook.com/groups/175356699802854/",
    "https://www.facebook.com/share/g/18JNQPA5PU/",
    "https://www.facebook.com/share/g/18NPYCEess/",
    "https://www.facebook.com/share/g/1EKebYgt5Q/",
    "https://www.facebook.com/share/g/185ytLwwFg/",
    "https://www.facebook.com/share/g/1Dv3yJSrK4/",
    "https://www.facebook.com/share/g/17G6mCkmKJ/",
    "https://www.facebook.com/share/g/1FkNPpjiQT/",
    "https://www.facebook.com/share/g/1CK38hkJKi/",
    "https://www.facebook.com/share/g/1GTKcMsvPH/",
    "https://www.facebook.com/share/g/16kde1JhfD/",
    "https://www.facebook.com/share/g/1GdLnRnNa8/",
    "https://www.facebook.com/share/g/14VETLLJxFk/",
    "https://www.facebook.com/share/g/183AURyK31/",
    "https://www.facebook.com/share/g/1ATGortt8h/",
    "https://www.facebook.com/share/g/1briWbWRkE/",
    "https://www.facebook.com/share/g/18PynzrLZs/",
    "https://www.facebook.com/share/g/1Am4xiU2gK/",
    "https://www.facebook.com/share/g/17DJKPBtHd/",
    "https://www.facebook.com/groups/propertydit/",
    "https://www.facebook.com/share/g/185RyKmRPE/",
    "https://www.facebook.com/share/g/1CY3Mbb748/",
    "https://www.facebook.com/share/g/1B8oxbjTvm/",
    "https://www.facebook.com/share/g/1GSXJfPHRv/",
    "https://www.facebook.com/share/g/1M1HGCPGcu/",
    "https://www.facebook.com/share/g/1TNPq3ht9y/",
    "https://www.facebook.com/groups/2234744070053568/",
    "https://www.facebook.com/share/g/1CQGfxHye9/",
    "https://www.facebook.com/share/g/1DSPbvtBao/",
    "https://www.facebook.com/share/g/1GSvzLRBLr/",
    "https://www.facebook.com/share/g/1F8jgQaN2A/",
    "https://www.facebook.com/share/g/1HbNgxFjuL/",
    "https://www.facebook.com/share/g/1C88zkj62s/",
    "https://www.facebook.com/share/g/1Av5UuLqoH/",
    "https://www.facebook.com/share/g/1SmN6nuyk2/",
    "https://www.facebook.com/share/g/17H6RfoLRU/",
    "https://www.facebook.com/share/g/1cLKmAvPxn/",
    "https://www.facebook.com/share/g/174RKdW4tb/",
    "https://www.facebook.com/share/g/19WUdWcuDX/",
    "https://www.facebook.com/share/g/1Y2vcMpTCi/",
    "https://www.facebook.com/share/g/1CE4tovrbk/",
    "https://www.facebook.com/groups/1756597841090354/",
    "https://www.facebook.com/groups/theoneestate/",
    "https://www.facebook.com/groups/1512942549024788/",
    "https://www.facebook.com/share/g/1N8fiVtKft/",
    "https://www.facebook.com/share/g/17148ebYu2/",
    "https://www.facebook.com/share/g/1MeEAycCK9/",
    "https://www.facebook.com/share/g/1SZj2CdzZG/",
    "https://www.facebook.com/share/g/1J6mpUjhH1/",
    "https://www.facebook.com/share/g/1FBhAHeU2C/",
    "https://www.facebook.com/share/g/18PtvBe5Rk/",
    "https://www.facebook.com/share/g/1CV291GDMX/",
    "https://www.facebook.com/share/g/1ExkvVre1Z/",
    "https://www.facebook.com/groups/210390739740584/",
    "https://www.facebook.com/share/g/1E4S1igybj/",
    "https://www.facebook.com/share/g/1CKJ9gZFiE/",
    "https://www.facebook.com/share/g/1EMvFAXfAG/",
    "https://www.facebook.com/share/g/1BT3jt6Gzc/",
    "https://www.facebook.com/share/g/1AbuUgw43X/",
    "https://www.facebook.com/share/g/1GZJbNyZ7W/",
    "https://www.facebook.com/share/g/18GwgqjiBc/",
    "https://www.facebook.com/share/g/1HqpNUtH8B/",
    "https://www.facebook.com/share/g/1GXwMupTDL/",
    "https://www.facebook.com/share/g/1JTxjApoKP/",
    "https://www.facebook.com/share/g/19VteKNLoD/",
    "https://www.facebook.com/share/g/1C6x2GoH7u/",
    "https://www.facebook.com/share/g/1DaoFJMJkP/",
    "https://www.facebook.com/share/g/1DGSJAvihX/",
    "https://www.facebook.com/share/g/1AjZE6ZfSd/",
    "https://www.facebook.com/share/g/1DQok3tTUH/",
    "https://www.facebook.com/share/g/1C6oYUM8aD/",
    "https://www.facebook.com/share/g/1GEPncCTh9/",
    "https://www.facebook.com/share/g/1DjbxnhQBA/",
    "https://www.facebook.com/share/g/18Jr24GQVM/",
    "https://www.facebook.com/share/g/186TqCt9nn/",
    "https://www.facebook.com/share/g/1LKtYvTQyb/",
    "https://www.facebook.com/share/g/1HcXDi1ghz/",
    "https://www.facebook.com/share/g/1C1PfcEEU9/",
    "https://www.facebook.com/share/g/18JoUsmbRP/",
    "https://www.facebook.com/share/g/16wHaXjF3W/",
    "https://www.facebook.com/share/g/17AZ9awNPP/",
    "https://www.facebook.com/groups/770357389784713/",
    "https://www.facebook.com/share/g/1CB7iUsP7j/",
    "https://www.facebook.com/groups/condomarket/",
    "https://www.facebook.com/share/g/18EVu9X5hn/",
    "https://www.facebook.com/share/g/1KJZowXjYW/",
    "https://www.facebook.com/share/g/1Akg539Dgx/",
    "https://www.facebook.com/share/g/1L5WbUZqNR/",
    "https://www.facebook.com/groups/914277069917114/",
    "https://www.facebook.com/share/g/18MYU47ZQy/",
    "https://www.facebook.com/groups/1516977068738748/",
    "https://www.facebook.com/share/g/18akkmMoGG/",
]

OUTPUT_HEADERS = [
    "วันที่โพส", "website", "ประเภท", "สถานะ", "ชื่อโครงการ",
    "ขนาด", "ราคา", "เขต", "Link", "เบอร์โทรศัพท์", "Line", "คำอธิบาย"
]

SYSTEM_PROMPT = """คุณคือ AI วิเคราะห์อสังหาริมทรัพย์
ดึงข้อมูลจากข้อความที่ให้มา และตอบกลับเป็น JSON เท่านั้น
{
"is_real_estate": true/false,
"is_owner": true/false,
"owner_confidence": 0.0,
"evidence_phrases": [],
"risk_flags": [],
"post_date_text": "ดึงข้อความวันที่ที่พบในข้อมูล",
"extracted":{
"property_type":"",
"rental_sale_status":"",
"project_name":"",
"district":"",
"size_text":"",
"price_text":"",
"price_value_thb":null,
"phone":"",
"line":"",
"description":""
}
}

=== OWNER vs AGENT CLASSIFICATION ===

ประเมินว่าโพสต์นี้เป็น "เจ้าของขายเอง" หรือ "นายหน้า/เอเจนท์"

การประเมินเจ้าของ:
- ประเมิน is_owner ว่า true ถ้าดูมีลักษณะเจ้าของขายเอง, false ถ้าดูเป็นนายหน้า/เอเจนท์/บริษัท
- ใช้บริบทละเอียด เช่น คำปฏิเสธนายหน้า, บุรุษที่หนึ่ง ("ผม/ฉัน/ดิฉัน/เรา"), โทนแคมเปญหลายยูนิต/โปรโมชัน, คำเชิงเอเจนซี (รับฝากขาย/นายหน้า/โบรกเกอร์/realty/estate/property), ชื่อที่ส่อบริษัท, จำนวนเบอร์โทรหลายเบอร์, ข้อความบ่งชี้บริษัทหรือนายหน้า
- ข้อความ "รับนายหน้า/ยินดีนายหน้า/co-broker" อาจพบในโพสต์เจ้าของได้ จึงไม่นับเป็นลบอัตโนมัติ
- owner_confidence คือระดับความมั่นใจว่าประกาศนี้เป็น "เจ้าของขายเอง" (is_owner=true) อยู่ในช่วง 0.0–1.0
- evidence_phrases คือ list ของประโยค/วลีที่พบในข้อความซึ่งใช้ประกอบการตัดสิน
- risk_flags คือ list ของ signal ที่บ่งชี้ว่าอาจเป็นนายหน้า

RULE-BASED AGENT SIGNALS: ถ้าเจอข้อความเหล่านี้ให้ถือว่าเป็น "นายหน้า/เอเจนท์" (is_owner=false) เป็นหลัก
รับฝากขาย, รับฝากปล่อยเช่า, บริการด้านอสังหา, บริการจัดหาที่อยู่อาศัย, ทีมงานพร้อมพาชม,
ยินดีให้คำปรึกษาด้านอสังหา, เรามีห้องอีกหลายแบบ, มีห้องให้เลือกหลายทำเล, พร้อมจัดหาทรัพย์ตามงบ,
บริการครบวงจร, บริการพาเช็คเครดิต, ยินดีพาชมทุกวัน, ทีมเซลล์พร้อมบริการ, บริษัทตัวแทนอสังหา,
Property Hub, Estate, Real Estate, Property Agent, Agent Service, ทีม, ตัวแทนขายโครงการ,
One Stop Service, ศูนย์รวมคอนโดเชียงใหม่, บริการด้านสินเชื่อ, ดูทรัพย์อื่นๆเพิ่มเติม,
ยินดีให้คำปรึกษาฟรี, ดันสินเชื่อทุกเคส, รวมพูลวิลล่าสวยๆในเชียงใหม่, teedd,
ยินดีให้คำปรึกษาด้านสินเชื่อกู้ซื้อบ้าน, ราคาเริ่มต้นที่, ราคาพิเศษเฉพาะเดือนนี้,
Hot deal, Promotion, มีหลายหลังให้เลือก, ศูนย์รวม

PATTERN เจ้าของจริง (ช่วยยืนยัน is_owner=true เมื่อไม่ขัดกับ RULE-BASED AGENT SIGNALS ด้านบน):

หมวด ภาษาพูดที่เจ้าของบ้านเท่านั้นที่ใช้:
เจ้าของขายเอง, เจ้าของ, Owner, รับนายหน้า, ไม่รับนายหน้า, งดรับนายหน้า, เจ้าของปล่อยเอง,
บ้านของผม, บ้านของแม่, อยู่เองมาก่อน, เพิ่งย้ายเลยปล่อยเช่า, เพิ่งรีโนเวทเอง,
ซื้อไว้นานแล้ว ไม่ค่อยได้อยู่, ของจริงสวยกว่าในรูป, ถ้าเลี้ยงสัตว์ต้องคุยกันก่อน,
ห้องนี้ผมดูแลเองตลอด, ขายเพราะต้องย้ายงาน, อยู่เองสะอาดมากครับ,
เสียดายมาก แต่ต้องขาย, ไม่ค่อยอยากขาย แต่ไม่มีคนอยู่, ตั้งใจทำไว้ดีมาก, ต้องกลับต่างจังหวัด

หมวด พฤติกรรมโพสต์ (ใช้ช่วยประกอบ ถ้าเห็นร่องรอยในข้อความ):
โพสต์แต่บ้านหลังเดิมซ้ำๆ, โพสต์ไม่บ่อย, มีข้อมูลที่เฉพาะบ้านหลังนี้เท่านั้น,
ไม่มีลายน้ำบริษัท ไม่มีโลโก้, รูปถ่ายแบบแสงไม่ดี มุมไม่สวย, เนื้อหาไม่ใช่สไตล์แคมเปญโฆษณา

กฎการตัดสินขั้นสุดท้าย:
- ถ้าพบ RULE-BASED AGENT SIGNALS → is_owner=false ทันที ไม่ว่าบริบทอื่นจะเป็นอย่างไร
- ถ้าพบ PATTERN เจ้าของจริง และไม่มี RULE-BASED AGENT SIGNALS → is_owner=true
- ถ้าไม่แน่ใจ → ใช้บริบทรวมตัดสิน และสะท้อนความไม่แน่ใจใน owner_confidence (ค่าต่ำ เช่น 0.4–0.6)
- ถ้าใน input มีฟิลด์ RULE_BASED_AGENT_PATTERNS ไม่ว่าง → is_owner=false เป็นค่าเริ่มต้น เว้นแต่บริบทชัดเจนมากว่าเป็นเจ้าของ
- ถ้าใน input มีฟิลด์ RULE_BASED_OWNER_PATTERNS ไม่ว่าง และ RULE_BASED_AGENT_PATTERNS ว่าง → is_owner=true เป็นค่าเริ่มต้น
- สะท้อน evidence_phrases และ risk_flags จาก pattern ที่ตรวจพบเสมอ
"""

MONTH_MAP = {
    "มกราคม": 1, "ม.ค.": 1, "กุมภาพันธ์": 2, "ก.พ.": 2, "มีนาคม": 3, "มี.ค.": 3,
    "เมษายน": 4, "เม.ย.": 4, "พฤษภาคม": 5, "พ.ค.": 5, "มิถุนายน": 6, "มิ.ย.": 6,
    "กรกฎาคม": 7, "ก.ค.": 7, "สิงหาคม": 8, "ส.ค.": 8, "กันยายน": 9, "ก.ย.": 9,
    "ตุลาคม": 10, "ต.ค.": 10, "พฤศจิกายน": 11, "พ.ย.": 11, "ธันวาคม": 12, "ธ.ค.": 12,
}

csv_lock = threading.Lock()


def get_chrome_version(chrome_exec: str) -> int:
    try:
        res = subprocess.run([chrome_exec, "--version"], capture_output=True, text=True, check=False)
        return int(re.search(r"(\d+)\.", res.stdout).group(1)) if res.stdout else 0
    except Exception:
        return 0


def create_driver() -> uc.Chrome:
    PROFILE_PATH.mkdir(parents=True, exist_ok=True)
    chrome_exec = shutil.which("google-chrome") or shutil.which("chromium-browser")
    opts = uc.ChromeOptions()
    prefs = {"profile.managed_default_content_settings.images": 2, "notifications": 2}
    opts.add_experimental_option("prefs", prefs)
    opts.add_argument(f"--user-data-dir={PROFILE_PATH}")
    opts.add_argument("--disable-notifications")
    opts.add_argument("--disable-gpu")
    opts.page_load_strategy = "eager"
    return uc.Chrome(options=opts, version_main=get_chrome_version(chrome_exec), browser_executable_path=chrome_exec)


def humanized_scroll(driver: uc.Chrome) -> None:
    driver.execute_script(f"window.scrollBy(0, {SCROLL_SIZE + random.randint(-500, 500)});")
    time.sleep(random.uniform(1.2, 2.5))
    if random.random() > 0.8:
        driver.execute_script("window.scrollBy(0, -300);")


def apply_new_post_filter(driver: uc.Chrome):
    driver.execute_script("""
        let t = Array.from(document.querySelectorAll('[role="button"]')).find(e => e.innerText.match(/เรียงลำดับ|เกี่ยวข้อง|จัดเรียง/));
        if(t) { t.click(); setTimeout(() => {
            let o = Array.from(document.querySelectorAll('[role="menuitemradio"]')).find(e => e.innerText.match(/โพสต์ใหม่|New posts/));
            if(o) o.click();
        }, 1000); }
    """)
    time.sleep(2.5)


def batch_extract_dom(driver: uc.Chrome) -> List[Dict[str, str]]:
    return driver.execute_script("""
        document.querySelectorAll('div[role="button"]').forEach(b => {
            if (['See more', 'ดูเพิ่มเติม'].includes((b.innerText||'').trim())) b.click();
        });
        let results = [];
        document.querySelectorAll("div[role='article']").forEach(a => {
            let linkNode = Array.from(a.querySelectorAll("a[href]")).find(l => {
                let h = l.href || "";
                return h.includes('/posts/') || h.includes('/permalink/');
            });
            if (!linkNode) return;
            let url = linkNode.href.split('?')[0];
            if (url.includes('/reel/') || url.includes('/watch/')) return;
            let msgNode = a.querySelector("div[data-ad-preview='message']");
            let content = msgNode ? msgNode.innerText.trim() : "";
            if (!content) return;
            let date = "N/A";
            let dateLinks = a.querySelectorAll("a[aria-label]");
            for (let d of dateLinks) {
                let label = (d.getAttribute("aria-label") || '').trim();
                if (label.length < 30 && /\\d/.test(label)) { date = label; break; }
            }
            results.push({"Post_URL": url, "Full_Content": content, "Date": date});
        });
        return results;
    """)


def normalize_json_string(text: str) -> str:
    cleaned = re.sub(r"^```.*?\n|\n```$", "", text.strip(), flags=re.S)
    start, end = cleaned.find("{"), cleaned.rfind("}")
    return cleaned[start : end + 1] if -1 < start < end else ""


def call_llm_service(payload: str) -> dict | None:
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                temperature=0.1,
                max_tokens=16000,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": payload},
                ],
                response_format={"type": "json_object"}
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            logger.warning(f"Retry {attempt + 1}: {str(e)}")
            time.sleep(2 ** attempt)
    return None


def parse_date(date_text: str) -> str:
    now = datetime.now()
    if not date_text or date_text == "-":
        return "-"
    val = date_text.strip().lower()
    if any(k in val for k in ("วันนี้", "นาที", "ชั่วโมง", "ชม.")):
        return now.strftime("%d/%m/%Y")
    if "เมื่อวาน" in val:
        return (now - timedelta(days=1)).strftime("%d/%m/%Y")
    m_days = re.search(r"(\d+)\s*วัน", val)
    if m_days:
        return (now - timedelta(days=int(m_days.group(1)))).strftime("%d/%m/%Y")
    m_date = re.search(r"(\d{1,2})[\s\.\-/]+([ก-๙a-zA-Z]+)(?:[\s\.\-/]+(\d{2,4}))?", val)
    if m_date:
        d, m_raw = int(m_date.group(1)), m_date.group(2)
        m = MONTH_MAP.get(m_raw)
        if m:
            y = now.year
            if m_date.group(3):
                y_raw = int(m_date.group(3))
                y = y_raw - 543 if y_raw > 2400 else (y_raw if y_raw > 100 else 2000 + y_raw)
            return f"{d:02d}/{m:02d}/{y}"
    return "-"


def transform_record(raw_row: dict, ai_data: dict) -> dict:
    ext = ai_data.get("extracted", {})
    return {
        "วันที่โพส": parse_date(ai_data.get("post_date_text", "")),
        "website": "facebook",
        "ประเภท": ext.get("property_type", "-"),
        "สถานะ": ext.get("rental_sale_status", "-"),
        "ชื่อโครงการ": ext.get("project_name", "-"),
        "ขนาด": ext.get("size_text", "-"),
        "ราคา": ext.get("price_text", "-"),
        "เขต": ext.get("district", "-"),
        "Link": raw_row.get("Post_URL") or raw_row.get("URL", "-"),
        "เบอร์โทรศัพท์": ext.get("phone", "-"),
        "Line": ext.get("line", "-"),
        "คำอธิบาย": ext.get("description", "-"),
    }


def worker_process_and_save(raw_item: dict) -> None:
    context = "\n".join([f"{k}: {v}" for k, v in raw_item.items() if v])[:2000]
    ai_response = call_llm_service(context)

    if not ai_response:
        return

    if not ai_response.get("is_real_estate"):
        return

    if ai_response.get("is_agent"):
        signals = ai_response.get("agent_signals", [])
        logger.info(f"AGENT_FILTERED | signals={signals} | url={raw_item.get('Post_URL', '')}")
        return

    final_data = transform_record(raw_item, ai_response)
    with csv_lock:
        with open(OUTPUT_PATH, 'a', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=OUTPUT_HEADERS)
            writer.writerow(final_data)


def process_group(driver: uc.Chrome, url: str, seen_urls: Set[str], executor: ThreadPoolExecutor):
    try:
        driver.get(url)
        time.sleep(3)
        apply_new_post_filter(driver)
    except Exception as e:
        logger.error(f"Failed loading {url}: {e}")
        return

    saved_count = 0
    stagnant_count = 0

    for _ in range(500):
        extracted = batch_extract_dom(driver)
        new_items = []

        for item in extracted:
            if item["Post_URL"] not in seen_urls:
                seen_urls.add(item["Post_URL"])
                new_items.append(item)
                saved_count += 1

        if new_items:
            stagnant_count = 0
            for item in new_items:
                executor.submit(worker_process_and_save, item)
            logger.info(f"Batched {len(new_items)} posts to LLM... (Total: {saved_count}/{TARGET_POSTS})")
        else:
            stagnant_count += 1

        if saved_count >= TARGET_POSTS or stagnant_count >= MAX_STAGNANT:
            break

        humanized_scroll(driver)

    logger.info(f"Group Done | Collected & Sent to LLM: {saved_count}")


def main():
    if not OUTPUT_PATH.exists() or OUTPUT_PATH.stat().st_size == 0:
        OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(OUTPUT_PATH, "w", newline="", encoding="utf-8-sig") as f:
            csv.DictWriter(f, fieldnames=OUTPUT_HEADERS).writeheader()

    seen_urls: Set[str] = set()
    with open(OUTPUT_PATH, "r", encoding="utf-8-sig") as f:
        seen_urls.update(row["Link"] for row in csv.DictReader(f) if row.get("Link"))

    driver = None
    try:
        driver = create_driver()
        driver.get("https://www.facebook.com/")
        time.sleep(3)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            for idx, group_url in enumerate(GROUP_URLS, 1):
                logger.info(f"Processing Group [{idx}/{len(GROUP_URLS)}]")
                process_group(driver, group_url, seen_urls, executor)
                time.sleep(random.uniform(2.0, 4.0))

    except Exception as e:
        logger.critical(f"Pipeline crashed: {e}")
    finally:
        if driver:
            driver.quit()


if __name__ == "__main__":
    main()

[22:55:24] [INFO] patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
[22:55:31] [INFO] Processing Group [1/158]
[22:55:40] [INFO] Batched 1 posts to LLM... (Total: 1/100)
[22:55:44] [INFO] Batched 1 posts to LLM... (Total: 2/100)
[22:55:46] [INFO] Batched 3 posts to LLM... (Total: 5/100)
[22:55:49] [INFO] Batched 3 posts to LLM... (Total: 8/100)
[22:55:49] [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
[22:55:51] [INFO] Batched 1 posts to LLM... (Total: 9/100)
[22:55:53] [INFO] Batched 1 posts to LLM... (Total: 10/100)
[22:55:56] [INFO] Batched 1 posts to LLM... (Total: 11/100)
[22:55:58] [INFO] Batched 2 posts to LLM... (Total: 13/100)
[22:56:00] [INFO] Batched 1 posts to LLM... (Total: 14/100)
[22:56:02] [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
[22:56:02] [INFO] Batched 2 posts to LLM... (Total: 16/100)
[22:56:03] [INFO] HTTP Request: POST ht